In [92]:
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup
import json
import msgpack
import pandas as pd
import time
import random
import regex as re

In [ ]:
# https://portal.dnb.de/opac.htm?method=simpleSearch&query=Monika+Wogrolly+Die+Menschenfresserin

In [123]:
def search_dnb(author, title):
    url = 'https://portal.dnb.de/opac.htm?method=simpleSearch&query='
    query = author + ' ' + title
    query= query.replace(' ', '+')
    filters = '&cqlMode=false&sortOrderIndex=jhr_asc&categoryId=books'
    html_text = requests.get(url+query+filters).text
    soup = BeautifulSoup(html_text, "html.parser")

    amount = soup.find('span', {'class': 'amount'})
    if not amount:
        return url+query+filters, 'Not found'
    if amount.text == 'Treffer 1 von 1':
        link_tag = soup.find('strong', string='Link zu diesem Datensatz').parent
        link = link_tag.find_next('td').text.strip()
        year_tag = soup.find('strong', string='Zeitliche Einordnung').parent
        year = year_tag.find_next('td').text.strip()
        if re.search(r'\d{4}', year):
            return url+query+filters, re.search(r'\d{4}', year).group(0)
    else:
        results = soup.find('table', {'class': 'searchresult'})
        if results:
            first_result = results.find('tr').find('a', {'title': 'Details zu diesem Datensatz'})
            if first_result.text.strip().lower().startswith(title.lower()):
                if re.search(r'\d{4}', first_result.text.strip()):
                    return url+query+filters, re.search(r'\d{4}', first_result.text.strip()).group(0)
            else:
                second_result = results.find('tr').find_next('tr').find('a', {'title': 'Details zu diesem Datensatz'})
                if second_result.text.strip().lower().startswith(title.lower()):
                    if re.search(r'\d{4}', second_result.text.strip()):
                        return url+query+filters, re.search(r'\d{4}', second_result.text.strip()).group(0)       
    return url+query+filters, 'Not found'
        



In [128]:
df = pd.read_excel('other.xlsx').fillna('')

years_lst = []
for _, row in tqdm(df.iterrows()):
    author = row['author']
    title = row['title']
    try:
        url, year = search_dnb(author, title)
    except KeyboardInterrupt:
        print('Process interrupted')
        break
    except Exception as e: 
        url = 'https://portal.dnb.de/opac.htm?method=simpleSearch&query='
        query = author + ' ' + title
        query= query.replace(' ', '+')
        filters = '&cqlMode=false&sortOrderIndex=jhr_asc&categoryId=books'
        url = url + query + filters
        years_lst.append((row['url'], author, title, 'Error', url))
        continue
    years_lst.append((row['url'], author, title, year, url))

4308it [54:34,  1.32it/s]


In [129]:
df_out = pd.DataFrame(years_lst, columns=['perlentaucher_url', 'author', 'title', 'dnb_year', 'dnb_query'])

df_out.to_excel('other_dnb_urls.xlsx', index=False)

In [ ]:
# Karl Mickel Lachmunds Freunde
search_dnb('Karl Mickel', 'Lachmunds Freunde')

('https://portal.dnb.de/opac.htm?method=simpleSearch&query=Karldsadsdasd+Mickel+Lachmunds+Freunde&cqlMode=false&sortOrderIndex=jhr_asc&categoryId=books',
 'Not found')

In [127]:
years_lst[:5]

[('https://www.perlentaucher.de/buch/erwin-marti/carl-albert-loosli-1877-1959-band-2.html',
  'Erwin Marti',
  'Carl Albert Loosli 1877 - 1959',
  'Not found',
  'https://portal.dnb.de/opac.htm?method=simpleSearch&query=Erwin+Marti+Carl+Albert+Loosli+1877+-+1959&cqlMode=false&sortOrderIndex=jhr_asc&categoryId=books'),
 ('https://www.perlentaucher.de/buch/thomas-karlauf/helmut-schmidt.html',
  'Thomas Karlauf',
  'Helmut Schmidt',
  'Not found',
  'https://portal.dnb.de/opac.htm?method=simpleSearch&query=Thomas+Karlauf+Helmut+Schmidt&cqlMode=false&sortOrderIndex=jhr_asc&categoryId=books'),
 ('https://www.perlentaucher.de/buch/monika-gisler/schoepferische-unruhe.html',
  'Monika Gisler',
  'Schöpferische Unruhe',
  '2025',
  'https://portal.dnb.de/opac.htm?method=simpleSearch&query=Monika+Gisler+Schöpferische+Unruhe&cqlMode=false&sortOrderIndex=jhr_asc&categoryId=books'),
 ('https://www.perlentaucher.de/buch/peter-kohl-dona-kujacinski/hannelore-kohl.html',
  '\nPeter Kohl, Dona Kujacinsk